<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/02_construction_data_preparation_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DADS5001 Mini Project
## การเตรียมข้อมูลรายการจัดซื้อจัดจ้างประเภทจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 **สะสมถึงวันที่ 30 กรกฎาคม 2569** ไม่ใช่ข้อมูลเต็มปี Notebook นี้อ่านไฟล์ต้นทาง 8 ไฟล์แบบแบ่งส่วน แล้วสกัดรายการประเภท `จ้างก่อสร้าง`

ใช้คำว่า **รายการ** กับข้อมูลต้นทาง เพราะหนึ่งโครงการอาจมีหลายแถว จากนั้นใช้ `รหัสโครงการ` สร้างหน่วยวิเคราะห์ระดับโครงการ โดยยังเก็บทุกแถวไว้สำหรับตรวจสัญญา ผู้รับจ้าง และ Joint Venture (JV)

ไฟล์ผลลัพธ์เก็บงานก่อสร้างทุกวิธีและทุกวงเงิน เพื่อให้ Notebook 03 สำรวจข้อมูลก่อนกำหนดกลุ่มศึกษาหลัก

> ขั้นตอนนี้เตรียมและตรวจสอบข้อมูล ยังไม่สร้างตัวชี้วัดหรือสรุปความผิด


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd

# ดาวน์โหลดฟอนต์ TH Sarabun New
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# เพิ่มฟอนต์ให้ Matplotlib
fm.fontManager.addfont(
    'thsarabunnew-webfont.ttf'
)

# กำหนดฟอนต์เริ่มต้นสำหรับ Matplotlib และ Seaborn
mpl.rc(
    'font',
    family='TH Sarabun New'
)
mpl.rcParams['axes.unicode_minus'] = False

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

base_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract/2569'
)

processed_dir = base_dir.parent / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(base_dir.glob('*.csv'))
construction_path = processed_dir / 'construction_contracts_2569.csv'

project_directory = base_dir.parents[3]
figure_directory = project_directory / 'figure'
figure_directory.mkdir(parents=True, exist_ok=True)

print(f'CSV files found: {len(csv_files)}')
print(f'Construction output: {construction_path}')
print(f'Figure directory: {figure_directory}')


## 1. สำรวจโครงสร้างข้อมูลเบื้องต้น

เริ่มจากอ่านข้อมูลตัวอย่างจากไฟล์แรก เพื่อทำความเข้าใจโครงสร้าง
ชื่อคอลัมน์ และลักษณะข้อมูล ก่อนประมวลผลไฟล์ทั้งหมด

In [ ]:
sample_data = pd.read_csv(
    csv_files[0],
    nrows=5
)

print(f'Sample file: {csv_files[0].name}')
print(f'Number of columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for position, column in enumerate(
    sample_data.columns,
    start=1
):
    print(f'{position:02d}. {column}')

## 2. ภาพรวมประเภทโครงการและการสกัดข้อมูล

อ่านข้อมูลแบบแบ่งส่วนเพื่อ:

1. นับจำนวนแถวตามประเภทโครงการ
2. สกัดรายการ `จ้างก่อสร้าง` ทุกคอลัมน์
3. เก็บชื่อไฟล์ต้นทางเพื่อให้ตรวจสอบย้อนกลับได้

ส่วนนี้ใช้คำว่า “จำนวนรายการ” เพราะหนึ่งโครงการอาจมีหลายสัญญา ผู้รับจ้าง หรือแถวสมาชิก Joint Venture (JV)


In [ ]:
project_type_column = 'ชื่อประเภทโครงการ'
chunk_size = 100_000

type_counts_list = []
processing_results = []

first_write = True

for file_number, file_path in enumerate(
    csv_files,
    start=1
):
    print(
        f'Processing {file_number}/{len(csv_files)}: '
        f'{file_path.name}'
    )

    total_rows = 0
    construction_rows = 0

    reader = pd.read_csv(
        file_path,
        chunksize=chunk_size,
        low_memory=False
    )

    for chunk in reader:
        chunk.columns = chunk.columns.str.strip()
        total_rows += len(chunk)

        project_type = (
            chunk[project_type_column]
            .astype('string')
            .str.strip()
            .fillna('ไม่ระบุ')
        )

        type_counts_list.append(
            project_type.value_counts()
        )

        construction_chunk = chunk.loc[
            project_type.eq('จ้างก่อสร้าง')
        ].copy()

        construction_rows += len(
            construction_chunk
        )

        if not construction_chunk.empty:
            construction_chunk['source_file'] = (
                file_path.name
            )

            construction_chunk.to_csv(
                construction_path,
                mode='w' if first_write else 'a',
                header=first_write,
                index=False,
                encoding='utf-8-sig'
            )

            first_write = False

    processing_results.append({
        'file_name': file_path.name,
        'total_rows': total_rows,
        'construction_rows': construction_rows
    })

    print(
        f'  Total rows: {total_rows:,} | '
        f'Construction rows: {construction_rows:,}'
    )

processing_summary = pd.DataFrame(
    processing_results
)

project_type_counts = (
    pd.concat(
        type_counts_list,
        axis=1
    )
    .fillna(0)
    .sum(axis=1)
    .astype('int64')
    .sort_values(ascending=False)
    .rename('record_count')
    .reset_index()
    .rename(
        columns={
            'index': project_type_column
        }
    )
)

total_records = (
    processing_summary['total_rows'].sum()
)

total_construction_records = (
    processing_summary[
        'construction_rows'
    ].sum()
)

construction_pct = (
    total_construction_records
    / total_records
    * 100
)

display(processing_summary)

print(f'Total records: {total_records:,}')
print(
    f'Construction records: '
    f'{total_construction_records:,}'
)
print(
    f'Construction share: '
    f'{construction_pct:.2f}%'
)
print(f'Output file: {construction_path}')

In [ ]:
# ตรวจยืนยันผลการประมวลผลกับข้อมูลชุดที่ใช้ในโครงการ
expected_file_count = 8
expected_total_records = 3_964_924
expected_construction_records = 180_079

assert len(csv_files) == expected_file_count, (
    f'Expected {expected_file_count} source files, '
    f'but found {len(csv_files)}'
)
assert total_records == expected_total_records, (
    f'Expected {expected_total_records:,} source rows, '
    f'but found {total_records:,}'
)
assert total_construction_records == expected_construction_records, (
    f'Expected {expected_construction_records:,} construction rows, '
    f'but found {total_construction_records:,}'
)
assert construction_path.exists(), (
    f'Output file was not created: {construction_path}'
)

print('Validation passed:')
print(f'- Source files: {len(csv_files)}')
print(f'- Source rows: {total_records:,}')
print(f'- Construction rows: {total_construction_records:,}')
print(f'- Output file exists: {construction_path.exists()}')

### เปรียบเทียบสัดส่วนประเภทโครงการ

หลังตรวจสอบยอดรวมจากไฟล์ทั้ง 8 ส่วนแล้ว จึงสรุปจำนวนรายการตามประเภทโครงการและสร้างกราฟเพื่อดูว่างานจ้างก่อสร้างมีขนาดเท่าใดเมื่อเทียบกับข้อมูลทั้งหมด

ผลจากตารางและกราฟจะใช้กำหนดขอบเขตข้อมูลก่อนตรวจว่าแต่ละรายการแทนโครงการ สัญญา หรือผู้รับจ้างในระดับใด


In [ ]:
project_type_counts['record_pct'] = (
    project_type_counts['record_count']
    .div(
        project_type_counts[
            'record_count'
        ].sum()
    )
    .mul(100)
)

display(project_type_counts)

In [ ]:
major_types = (
    project_type_counts
    .head(4)
    .sort_values('record_pct', ascending=True)
    .copy()
)

colors = [
    '#E67E22'
    if project_type == 'จ้างก่อสร้าง'
    else '#8FA3B8'
    for project_type
    in major_types['ชื่อประเภทโครงการ']
]

fig, ax = plt.subplots(figsize=(11, 5.5))

bars = ax.barh(
    major_types['ชื่อประเภทโครงการ'],
    major_types['record_pct'],
    color=colors
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} รายการ ({percentage:.2f}%)'
        for count, percentage in zip(
            major_types['record_count'],
            major_types['record_pct']
        )
    ],
    padding=4,
    fontsize=10
)

ax.set_title(
    'ข้อมูลจ้างก่อสร้างคิดเป็น 4.54% ของรายการจัดซื้อจัดจ้างทั้งหมด'
)
ax.set_xlabel('สัดส่วนของจำนวนรายการ (%)')
ax.set_ylabel('ประเภทโครงการ')
ax.set_xlim(0, major_types['record_pct'].max() * 1.22)
ax.text(
    0,
    -0.18,
    f'ฐานข้อมูลทั้งหมด {project_type_counts["record_count"].sum():,.0f} รายการ',
    transform=ax.transAxes,
    fontsize=10,
    color='#555555'
)
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = (
    figure_directory
    / 'fig02_01_share_of_procurement_records.png'
)
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


### สิ่งที่พบและคำถามถัดไป

งานจ้างก่อสร้างเป็น 180,079 จาก 3,964,924 รายการ หรือ 4.54% ของข้อมูลทั้งหมด ภาพนี้แสดงสัดส่วนของ **จำนวนแถว** ไม่ใช่สัดส่วนจำนวนโครงการหรือมูลค่า

ขั้นต่อไปต้องจัดข้อมูลเป็นหนึ่งแถวต่อโครงการ ก่อนสำรวจขนาดโครงการ วิธีจัดซื้อ และเส้นวงเงิน 500,000 บาท


## 3. ตรวจสอบชุดข้อมูลจ้างก่อสร้าง

ตรวจจำนวนแถว คอลัมน์ รหัสโครงการ แถวซ้ำ และโครงการที่ปรากฏหลายแถว ก่อนส่งต่อไปสร้างข้อมูลระดับโครงการและระดับสัญญา


In [ ]:
construction_data = pd.read_csv(
    construction_path,
    low_memory=False
)

print(
    f'Shape: '
    f'{construction_data.shape}'
)

display(
    construction_data.head()
)

In [ ]:
validation_summary = pd.Series({
    'Rows': len(construction_data),
    'Columns': construction_data.shape[1],
    'Unique project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].nunique()
    ),
    'Unique contract number labels (not contract count)': (
        construction_data[
            'เลขที่สัญญา'
        ].nunique()
    ),
    'Missing project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].isna().sum()
    ),
    'Missing contract numbers': (
        construction_data[
            'เลขที่สัญญา'
        ].isna().sum()
    ),
    'Exact duplicate rows': (
        construction_data
        .drop(columns='source_file')
        .duplicated()
        .sum()
    )
})

display(
    validation_summary.to_frame(
        name='value'
    )
)

display(
    construction_data[
        'ชื่อประเภทโครงการ'
    ].value_counts(
        dropna=False
    )
)

In [ ]:
project_row_counts = (
    construction_data[
        'รหัสโครงการ'
    ]
    .value_counts()
)

print(
    'Projects with more than one row:',
    (project_row_counts > 1).sum()
)

print(
    'Maximum rows per project:',
    project_row_counts.max()
)

print(
    '\nMost frequent contract number labels '
    '(not a count of unique contracts):'
)

display(
    construction_data[
        'เลขที่สัญญา'
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)

repeated_project_ids = (
    project_row_counts.loc[
        project_row_counts > 1
    ]
    .head(3)
    .index
)

repeated_project_sample = (
    construction_data.loc[
        construction_data[
            'รหัสโครงการ'
        ].isin(
            repeated_project_ids
        ),
        [
            'รหัสโครงการ',
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            'ชื่อผู้ชนะการเสนอราคา',
            'เลขที่สัญญา',
            'วงเงินงบประมาณในสัญญา (บาท)'
        ]
    ]
    .sort_values(
        [
            'รหัสโครงการ',
            'เลขที่สัญญา'
        ]
    )
    .head(20)
)

display(repeated_project_sample)

### ตรวจคุณภาพพื้นที่และหน่วยวิเคราะห์ผู้รับจ้าง

ก่อนส่งข้อมูลไปวิเคราะห์ Pattern เชิงพื้นที่ ต้องตรวจว่าหนึ่งโครงการเชื่อมกับจังหวัดกี่ค่า และคู่ `โครงการ–ผู้รับจ้าง` มีข้อมูลครบเพียงใด

ส่วนนี้ตอบเฉพาะคุณภาพและความพร้อมของข้อมูล ยังไม่ใช้การกระจุกของจังหวัดหรือผู้รับจ้างเป็นข้อสรุปความผิดสังเกต


In [ ]:
project_id_column = 'รหัสโครงการ'
province_column = 'จังหวัด'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'

construction_data[province_column] = (
    construction_data[province_column]
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

province_count_by_project = (
    construction_data
    .groupby(project_id_column, dropna=False)[province_column]
    .nunique(dropna=True)
)

project_supplier_pairs = (
    construction_data[
        [
            project_id_column,
            supplier_id_column,
            supplier_name_column,
            province_column
        ]
    ]
    .drop_duplicates()
)

area_supplier_quality = pd.Series({
    'โครงการทั้งหมด': construction_data[project_id_column].nunique(),
    'โครงการที่จังหวัดหาย': (
        construction_data
        .groupby(project_id_column)[province_column]
        .first()
        .isna()
        .sum()
    ),
    'โครงการที่พบมากกว่า 1 จังหวัด': province_count_by_project.gt(1).sum(),
    'คู่ project–supplier ไม่ซ้ำ': len(
        project_supplier_pairs[
            [project_id_column, supplier_id_column]
        ].drop_duplicates()
    ),
    'คู่ project–supplier ที่รหัสผู้รับจ้างหาย': (
        project_supplier_pairs[supplier_id_column].isna().sum()
    )
}, name='value')

display(area_supplier_quality.to_frame())

province_spelling_check = (
    construction_data[province_column]
    .value_counts(dropna=False)
    .rename_axis(province_column)
    .reset_index(name='record_count')
)

display(province_spelling_check)


### วิธีอ่านผลตรวจคุณภาพพื้นที่

หลังรัน ให้ตรวจจำนวนจังหวัดและรายการสะกดที่ผิดรูปก่อนวิเคราะห์ หากโครงการเดียวพบมากกว่าหนึ่งจังหวัด ต้องย้อนดูแถวต้นทางแทนการเลือกจังหวัดโดยอัตโนมัติ ส่วนค่าจังหวัดหรือรหัสผู้รับจ้างที่หายต้องรายงานเป็นข้อมูลที่ถูกตัดออกจาก Pattern เชิงพื้นที่


In [ ]:
# Confirm that the study scope can be reproduced from project-level fields
project_scope_data = (
    construction_data
    .drop_duplicates(
        subset='รหัสโครงการ',
        keep='first'
    )
    .copy()
)

study_scope_mask = (
    project_scope_data[
        'ชื่อวิธีการจัดซื้อจัดจ้าง'
    ].eq('เฉพาะเจาะจง')
    & project_scope_data[
        'วงเงินงบประมาณ (บาท)'
    ].le(500_000)
)

study_scope_summary = pd.Series({
    'โครงการจ้างก่อสร้างทั้งหมด': len(project_scope_data),
    'โครงการวิธีเฉพาะเจาะจงไม่เกิน 500,000 บาท': (
        study_scope_mask.sum()
    ),
    'สัดส่วนของโครงการก่อสร้างทั้งหมด (%)': (
        study_scope_mask.mean() * 100
    )
}, name='value')

display(study_scope_summary)

assert project_scope_data['รหัสโครงการ'].is_unique
assert study_scope_mask.sum() > 0

print('Study-scope validation passed')


### ผลลัพธ์ที่ส่งต่อ

หลังรัน Notebook นี้ ให้สรุปจำนวนรายการ จำนวนโครงการไม่ซ้ำ ความครบถ้วนของจังหวัด และคู่ `project–supplier` จาก Output ด้านบน

ไฟล์ `construction_contracts_2569.csv` เก็บงานก่อสร้างทุกวิธีและทุกวงเงิน เพื่อใช้สร้างข้อมูลระดับโครงการ ตรวจหลายสัญญา จัดการ JV และเปรียบเทียบข้อมูลสองฝั่งของเส้น 500,000 บาทใน Notebook ถัดไป

Notebook นี้ทำหน้าที่เตรียมและตรวจคุณภาพข้อมูล จึงยังไม่สร้าง Pattern หรือกล่าวถึงความผิดปกติของหน่วยงาน ผู้รับจ้าง หรือพื้นที่
